In [ ]:
# Dependancies for the preprocessingof the data

from pathlib import Path
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter,RecursiveCharacterTextSplitter,Language
from langchain_core.documents import Document

In [ ]:
# finding the PDFs in the local files

location=Path("data/policies")

""" 
rglob (Recursive): Searches the current directory and all subdirectories underneath it. 
glob (Non-recursive by default): Searches only in the current directory you call it on.
"""
pdf_files=list(location.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs to process.\n")

Found 17 PDFs to process.



## PARSING

In [6]:
# markdown_splitter & text_splitter
"""
MarkdownHeaderTextSplitter(...): Initializes the LangChain splitter configured to break markdown text into separate 
                                 Document objects based on structural headers rather than character count.

RecursiveCharacterTextSplitter: LangChain's primary text splitter. It splits text using a list of 
                                separators in order until chunks are small enough.
"""

headers=[("#", "Header 1"),("##", "Header 2"),("###", "Header 3"),] 

# for retaing the title in the markdown
markdown_splitter=MarkdownHeaderTextSplitter(headers_to_split_on=headers,
                                               strip_headers=False  
                                               )

# For chunking
text_splitter=RecursiveCharacterTextSplitter.from_language(language=Language.MARKDOWN,
                                                             chunk_size=1200,      
                                                             chunk_overlap=150
                                                             )

In [7]:
all_docs = []

# Process each PDF file step-by-step
for idx, pdf_path in enumerate(pdf_files, start=1):
    insurer_name = pdf_path.stem.replace("_", " ").title()
    print(f"[{idx}/{len(pdf_files)}] Parsing: {pdf_path.name}...")

    # Extract Markdown page-by-page to retain accurate page numbers
    page_data=pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)

    for page_dict in page_data:
        page_num = page_dict["metadata"]["page_number"]  
        page_text = page_dict["text"]

        if not page_text.strip():
            continue

        header_splits = markdown_splitter.split_text(page_text) # Split by Markdown headers first

        final_splits = text_splitter.split_documents(header_splits) # Chunker

        # C. Attach file and page metadata
        for doc in final_splits:
            doc.metadata.update({
                "source": pdf_path.name,
                "insurer": insurer_name,
                "page": page_num
            })
            all_docs.append(doc)

print(f"Processing Complete! Total Chunks Extracted: {len(all_docs)}")

[1/17] Parsing: Care_senior_health.pdf...
[2/17] Parsing: Care_supreme.pdf...
[3/17] Parsing: Care_ultimate_joy.pdf...
[4/17] Parsing: HDFC_equicover_health.pdf...
[5/17] Parsing: HDFC_pradhan_mantri_suraksha_bima_yojana.pdf...
[6/17] Parsing: HDFC_Women_Suraksha.pdf...
[7/17] Parsing: Star_Comprehensive_policy.pdf...
[8/17] Parsing: Star_Medi_Classic_Insurance_Policy.pdf...
[9/17] Parsing: Star_senior_citizen_policy.pdf...
[10/17] Parsing: Star_women_policy.pdf...
[11/17] Parsing: UI_Commercial_vehicles_3rd_party.pdf...
[12/17] Parsing: UI_Commercial_vehicles_cover.pdf...
[13/17] Parsing: UI_private_bike_1yr_OD_3yr_TP.pdf...
[14/17] Parsing: UI_private_bike_1yr_OD_5yr_TP.pdf...
[15/17] Parsing: UI_private_bike_only_TP.pdf...
[16/17] Parsing: UI_private_car_only_TP.pdf...
[17/17] Parsing: UI_Scooter_Own_Damage.pdf...
Processing Complete! Total Chunks Extracted: 1695


In [ ]:
# Contect(sample)

print(all_docs[1480].page_content)

## LIABILITY TO THIRD PARTIES  
1. Subject to the Limit of liability as laid down in the schedule hereto, the Company will indemnify the insured in the event of accident caused by or arising out of the use of the Motor Vehicle anywhere in India against all sums including claimant's costs and expenses which the insured shall become legally liable to pay in respect of  
- i. death of or bodily injury to any person so far as it is necessary to meet the requirements of the Motor Vehicles Act.  
- ii. damage to property other than property belonging to the insured or held in trust or in the custody or control of the insured up to the limit specified in the schedule.  
2.  
- The Company will also pay all costs  and expenses incurred with its written consent.  
3. In terms of and subject to the limitations of the indemnity which is granted by this policy to the insured, the Company will indemnify any driver who is driving the Motor Vehicle on the insured's order or with insured's permission 

In [ ]:
# Metadata(sample)

print(all_docs[1480].metadata)

{'Header 1': '**United India Insurance Company Limited Regd & Head office: 24, Whites Road, Chennai 600014 CIN: U93090TN1938GOI000108**', 'Header 2': 'LIABILITY TO THIRD PARTIES', 'source': 'UI_Commercial_vehicles_3rd_party.pdf', 'insurer': 'Ui Commercial Vehicles 3Rd Party', 'page': 1}


In [29]:
print(all_docs[1480])

page_content='## LIABILITY TO THIRD PARTIES  
1. Subject to the Limit of liability as laid down in the schedule hereto, the Company will indemnify the insured in the event of accident caused by or arising out of the use of the Motor Vehicle anywhere in India against all sums including claimant's costs and expenses which the insured shall become legally liable to pay in respect of  
- i. death of or bodily injury to any person so far as it is necessary to meet the requirements of the Motor Vehicles Act.  
- ii. damage to property other than property belonging to the insured or held in trust or in the custody or control of the insured up to the limit specified in the schedule.  
2.  
- The Company will also pay all costs  and expenses incurred with its written consent.  
3. In terms of and subject to the limitations of the indemnity which is granted by this policy to the insured, the Company will indemnify any driver who is driving the Motor Vehicle on the insured's order or with insured

In [34]:
unique_insurers = sorted(list({doc.metadata.get("insurer") for doc in all_docs if doc.metadata.get("insurer")}))

In [35]:
unique_insurers

['Care Senior Health',
 'Care Supreme',
 'Care Ultimate Joy',
 'Hdfc Equicover Health',
 'Hdfc Pradhan Mantri Suraksha Bima Yojana',
 'Hdfc Women Suraksha',
 'Star Comprehensive Policy',
 'Star Medi Classic Insurance Policy',
 'Star Senior Citizen Policy',
 'Star Women Policy',
 'Ui Commercial Vehicles 3Rd Party',
 'Ui Commercial Vehicles Cover',
 'Ui Private Bike 1Yr Od 3Yr Tp',
 'Ui Private Bike 1Yr Od 5Yr Tp',
 'Ui Private Bike Only Tp',
 'Ui Private Car Only Tp',
 'Ui Scooter Own Damage']

# Embedding & FAISS

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [22]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1916.80it/s]


In [30]:
from langchain_community.vectorstores import FAISS

print("Generating vector embeddings and saving local FAISS index...")

vectorstore = FAISS.from_documents(all_docs, embeddings) # Vectorize all chunks and store in FAISS database

vectorstore.save_local("faiss_index") # Save index to local folder

print("'faiss_index/' directory created with index.faiss and index.pkl!")

Generating vector embeddings and saving local FAISS index...
'faiss_index/' directory created with index.faiss and index.pkl!
